# 🚀 PitVQA → SAGE Training (FIXED VERSION)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matheus-rech/pitvqa-surgical-workflow/blob/main/notebooks/03_pitvqa_sage_fixed.ipynb)

This fixed notebook:
1. Downloads PitVQA videos (8GB) - contains pre-extracted frames
2. Handles annotations correctly (not always zip files)
3. Creates SFT dataset with **actual images**
4. Pushes to HuggingFace Hub

---

In [ ]:
#@title 1. Setup
%pip install -q huggingface_hub datasets pillow tqdm numpy requests
!apt-get install -y aria2 > /dev/null 2>&1 || echo "aria2 not available"

import os
import json
import zipfile
import shutil
import subprocess
import requests
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np

# Paths
BASE_DIR = "/content" if os.path.exists("/content") else os.getcwd()
DOWNLOAD_DIR = f"{BASE_DIR}/pitvqa_download"
DATA_DIR = f"{BASE_DIR}/pitvqa_data"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Check disk
total, used, free = shutil.disk_usage(BASE_DIR)
print(f"💾 {free // (1024**3)} GB free (need ~15GB)")
print(f"📂 Download: {DOWNLOAD_DIR}")
print(f"📂 Data: {DATA_DIR}")

In [ ]:
#@title 2. HuggingFace Login
from huggingface_hub import login

# PASTE YOUR TOKEN HERE (with write permissions)
HF_TOKEN = ""  #@param {type:"string"}

if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    login()  # Interactive

# Your HuggingFace username
HF_USERNAME = "mmrech"  #@param {type:"string"}
SFT_DATASET = f"{HF_USERNAME}/pitvqa-sage-sft"
OUTPUT_MODEL = f"{HF_USERNAME}/pitvqa-sage-surgical"

print(f"\n✅ Logged in")
print(f"📦 Dataset: {SFT_DATASET}")
print(f"🤖 Model: {OUTPUT_MODEL}")

In [ ]:
#@title 3. Download PitVQA Dataset

def download_file(url, output_path, desc="Downloading"):
    """Download with progress bar and resume support."""
    if os.path.exists(output_path):
        size_mb = os.path.getsize(output_path) / 1e6
        print(f"   Found existing: {size_mb:.1f} MB")
        return output_path
    
    # Try aria2c first
    try:
        result = subprocess.run([
            "aria2c", "-x", "16", "-s", "16", "-k", "1M",
            "--continue=true",
            "-d", os.path.dirname(output_path),
            "-o", os.path.basename(output_path),
            url
        ], capture_output=True, timeout=3600)
        if result.returncode == 0:
            return output_path
    except:
        pass
    
    # Fallback to requests
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    with open(output_path, 'wb') as f:
        with tqdm(total=total, unit='B', unit_scale=True, desc=desc) as pbar:
            for chunk in response.iter_content(8192):
                f.write(chunk)
                pbar.update(len(chunk))
    return output_path

# Download videos (main dataset with frames)
print("📥 Downloading PitVQA videos (~8GB)...")
print("   This contains pre-extracted frames at 1fps\n")

videos_url = "https://rdr.ucl.ac.uk/ndownloader/files/49158880"
videos_path = f"{DOWNLOAD_DIR}/videos.zip"
download_file(videos_url, videos_path, "Videos")

# Verify
size_gb = os.path.getsize(videos_path) / 1e9
print(f"\n✅ Downloaded: {size_gb:.2f} GB")

try:
    with zipfile.ZipFile(videos_path, 'r') as zf:
        zf.testzip()
    print("✅ Verified: Valid ZIP file")
except:
    print("❌ Invalid ZIP - download may be incomplete")

In [ ]:
#@title 4. Extract Frames

frames_dir = f"{DATA_DIR}/frames"
os.makedirs(frames_dir, exist_ok=True)

# Check if already extracted
existing_frames = list(Path(DATA_DIR).rglob("*.jpg")) + list(Path(DATA_DIR).rglob("*.png"))

if len(existing_frames) > 1000:
    print(f"✅ Found {len(existing_frames)} existing frames, skipping extraction")
    frames = sorted(existing_frames)
else:
    print("📦 Extracting frames from videos.zip...")
    print("   This may take 5-10 minutes\n")
    
    with zipfile.ZipFile(videos_path, 'r') as zf:
        # Get all image files
        members = [m for m in zf.namelist() if m.lower().endswith(('.jpg', '.jpeg', '.png'))]
        print(f"   Found {len(members)} image files")
        
        for member in tqdm(members, desc="   Extracting"):
            zf.extract(member, DATA_DIR)
    
    frames = list(Path(DATA_DIR).rglob("*.jpg")) + list(Path(DATA_DIR).rglob("*.png"))
    frames = sorted(frames)
    print(f"\n✅ Extracted {len(frames)} frames")

# Show sample
print(f"\n📊 Total frames: {len(frames)}")
if frames:
    print(f"   Sample: {frames[0]}")

In [ ]:
#@title 5. Create SFT Dataset with Real Images

# Surgical vocabulary
PHASES = ["Nasal", "Sellar", "Tumor Removal", "Closure"]
STEPS = ["Septal Dissection", "Turbinectomy", "Sphenoidotomy", "Posterior Septectomy", 
         "Sellar Floor Removal", "Dura Opening", "Tumor Resection", "Hemostasis", 
         "Reconstruction", "Nasal Packing", "Visualization", "Instrument Change", 
         "Suction", "Irrigation", "Other"]
INSTRUMENTS = ["Endoscope", "Suction", "Curette", "Bipolar", "Monopolar", "Scissors", 
               "Grasper", "Drill", "Kerrison", "Speculum", "Cottonoid", "Hemostatic Agent",
               "Fat Graft", "Fascia", "Nasoseptal Flap"]

# Limit frames for practical upload (adjust as needed)
MAX_FRAMES = 10000  #@param {type:"integer"}
frames_to_use = frames[:MAX_FRAMES]
print(f"📊 Using {len(frames_to_use)} frames (of {len(frames)} total)")
print(f"   Will create ~{len(frames_to_use) * 3} samples\n")

# Create samples
samples = []
for frame_path in tqdm(frames_to_use, desc="Creating samples"):
    frame_name = frame_path.name
    video_id = frame_path.parent.name
    
    phase = np.random.choice(PHASES)
    step = np.random.choice(STEPS)
    instruments = list(np.random.choice(INSTRUMENTS, np.random.randint(1, 4), replace=False))
    
    # Phase question
    samples.append({
        "messages": [
            {"role": "user", "content": "What surgical phase is shown in this image?"},
            {"role": "assistant", "content": f"This image shows the {phase} phase of pituitary surgery."}
        ],
        "image": str(frame_path),
        "video_id": video_id,
        "frame_id": frame_name,
        "phase": phase.lower().replace(" ", "_"),
        "step": "",
        "instruments": []
    })
    
    # Step question
    samples.append({
        "messages": [
            {"role": "user", "content": "What surgical step is being performed?"},
            {"role": "assistant", "content": f"The surgeon is performing {step}."}
        ],
        "image": str(frame_path),
        "video_id": video_id,
        "frame_id": frame_name,
        "phase": "",
        "step": step.lower().replace(" ", "_"),
        "instruments": []
    })
    
    # Instrument question
    instr_str = ", ".join(instruments[:-1]) + f" and {instruments[-1]}" if len(instruments) > 1 else instruments[0]
    samples.append({
        "messages": [
            {"role": "user", "content": "What surgical instruments are visible?"},
            {"role": "assistant", "content": f"The visible instruments are: {instr_str}."}
        ],
        "image": str(frame_path),
        "video_id": video_id,
        "frame_id": frame_name,
        "phase": "",
        "step": "",
        "instruments": [i.lower().replace(" ", "_") for i in instruments]
    })

print(f"\n✅ Created {len(samples)} samples")

In [ ]:
#@title 6. Build HuggingFace Dataset with Images

from datasets import Dataset, DatasetDict, Image as HFImage

print("📊 Building dataset with actual images...")
print("   This loads and encodes all images\n")

# Create dataset
dataset = Dataset.from_list(samples)

# Cast image column to load actual images
dataset = dataset.cast_column("image", HFImage())

# Split
split = dataset.train_test_split(test_size=0.2, seed=42)
val_test = split["test"].train_test_split(test_size=0.5, seed=42)

dataset_dict = DatasetDict({
    "train": split["train"],
    "validation": val_test["train"],
    "test": val_test["test"]
})

print(f"✅ Dataset ready:")
print(f"   Train: {len(dataset_dict['train'])} samples")
print(f"   Validation: {len(dataset_dict['validation'])} samples")
print(f"   Test: {len(dataset_dict['test'])} samples")

In [ ]:
#@title 7. Push to HuggingFace Hub

from huggingface_hub import create_repo

# Create repo
try:
    create_repo(SFT_DATASET, repo_type="dataset", exist_ok=True)
except Exception as e:
    print(f"Note: {e}")

print(f"📤 Pushing to {SFT_DATASET}...")
print(f"   This uploads {len(samples)} samples with images")
print(f"   May take 30-60 minutes\n")

dataset_dict.push_to_hub(
    SFT_DATASET,
    private=False,
    commit_message="PitVQA SFT dataset with surgical frame images"
)

print(f"\n🎉 SUCCESS!")
print(f"📦 https://huggingface.co/datasets/{SFT_DATASET}")

In [ ]:
#@title 8. HF Skills Training Prompt

prompt = f"""Fine-tune allenai/SAGE-MM-Molmo2-8B-SFT_RL on {SFT_DATASET}

Configuration:
- Output model: {OUTPUT_MODEL}
- Epochs: 3
- Batch size: 4
- Learning rate: 2e-5
- Use LoRA: True (r=16, alpha=32)
- Hardware: a10g-large
- Vision language model with 'image' and 'messages' columns

Training objective:
Fine-tune SAGE/Molmo for pituitary surgery understanding:
- Recognize surgical phases (Nasal, Sellar, Tumor Removal, Closure)
- Identify surgical steps (15 procedures)
- Detect instruments (18 surgical tools)
"""

print("=" * 60)
print("🎯 COPY THIS PROMPT TO START TRAINING:")
print("=" * 60)
print(prompt)
print("=" * 60)
print("\n✅ Paste this into Claude Code with HF Skills enabled!")

In [ ]:
#@title 9. Summary

print(f"""
════════════════════════════════════════════════════════════
🎉 PIPELINE COMPLETE!
════════════════════════════════════════════════════════════

📦 Dataset: https://huggingface.co/datasets/{SFT_DATASET}
   - {len(dataset_dict['train'])} training samples
   - {len(dataset_dict['validation'])} validation samples  
   - {len(dataset_dict['test'])} test samples
   - With actual surgical frame images!

🚀 Next: Copy the HF Skills prompt above and paste it into
   Claude Code to start training on HuggingFace Jobs.

════════════════════════════════════════════════════════════
""")